In [2]:
import os
import pickle
import re

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import myutils


In [9]:
workspace,experiment=myutils.workspace()
IS_FINE_TUNED = True
save = False
run1 = "pred_ms-2026-10-14_132744"
run2 = "pred_ms-2026-10-13_133232"

In [5]:
match1 = re.match(r'pred_(.+?)-(.+)_.*',run1)
match2 = re.match(r'pred_(.+?)-(.+)_.*',run2)
dataset = match1.group(1)
assert match2.group(1) == dataset
pred = 'pred-with-fine-tuning' if IS_FINE_TUNED else 'pred-wo-fine-tuning'
pred_dir = experiment / f'results/{pred}'

if (dataset == "pbmc3k"):
    data_file = data_dir = experiment/ "data/pbmc3k_scanpy_annotated_batches.h5ad"
if (dataset == "ms"):
    data_file = workspace / "data/ms/ms_batches.h5ad"

assert data_file.exists()

In [6]:
adata = sc.read(data_file)
adata_test = adata[adata.obs.str_batch == '1']
celltypes = list(adata.obs["celltype"].unique())

if 'X_umap' not in adata_test.obsm and 'umap' not in adata_test.obsm:
    sc.pp.neighbors(adata_test, n_neighbors=20, n_pcs=30)
    sc.tl.umap(adata_test)

In [7]:
def get_cm(pred, celltypes) :
    id2type = pred['id_maps']
    predictions = pred['predictions']
    labels = pred['labels']

    from sklearn.metrics import confusion_matrix

    ct = [id2type[i] for i in range(len(celltypes))]
    for i in {id2type[p] for p in predictions}:
        if i not in ct:
            ct.remove(i)
    cm = confusion_matrix(labels, predictions)
    cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    cm = pd.DataFrame(cm, index=ct[:cm.shape[0]], columns=ct[:cm.shape[0]])
    return cm

results = {}
for d in [run1, run2]:
    save_run = pred_dir / d
    results_file = save_run / 'results.pkl'
    if results_file.exists(): 
        results[d] = {'file':results_file}
        with open(results_file, 'rb') as f:
            pred = pickle.load(f)
        results[d]['results'] = pred
        results[d]['cm'] = get_cm(pred, celltypes)

In [14]:
comp = results[run1]['cm'] - results[run1]['cm']
comp.sum().sum()

0.0